In [60]:
import os, sys
import duckdb
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [61]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("OndasCalor")

In [62]:
PROJECT_PATH    = os.getcwd()
ROOT_DATA_PATH  = "C:\\Marco Conti\\Projetos\\Dados\\"

In [63]:
def write_data(df_, write_path, prefix_file_name):
    # df_temperatura_final.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_t2m_temperatura.csv", index=False)
    df_.toPandas().to_parquet(f"{write_path}\\{prefix_file_name}.parquet")

# Somente para processamento LOCAL, pode ser descartada
def write_data_csv(df_write, write_path, file_name):
    df_write.toPandas().to_csv(f"{write_path}\{file_name}", index=False)


def write_data_parquet_by_duck(df_write, write_path, file_name):
    # Convertendo o PySpark DataFrame para Arrow e salvando via DuckDB
    arrow_table = df_write.toArrow()
    duckdb.query(
        f"COPY arrow_table TO '{write_path}\\{file_name}' (FORMAT PARQUET, COMPRESSION ZSTD)"
    )

Seleciona os dados de temperatura para cada município - Periodo 1995 até 2026

In [64]:
df_temperatura = \
    spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\temperaturas_por_municipios_SEM_georrefenciamento.parquet")

In [65]:
# Adiciona as colunas de indicador e unidade_medida para manter o igualdade de colunas entre os próximos dataframes
df_temperatura = \
    df_temperatura.withColumns({"indicador":      F.lit("Temperatura diaria °C ")
                               ,"unidade_medida": F.lit("Celsius") 
                               })

df_temperatura.printSchema()
print("df_temperatura.count():", df_temperatura.count())
# df_temperatura.filter("name_muni = 'São Paulo' and data_medicao = '2025-07-01'").show(100,False)


# data_medicao|latitude|longitude|  indicador|             valor|unidade_medida

root
 |-- code_muni: double (nullable = true)
 |-- name_muni: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- ANO: long (nullable = true)
 |-- temp_media_municipio: double (nullable = true)
 |-- indicador: string (nullable = false)
 |-- unidade_medida: string (nullable = false)

df_temperatura.count(): 63676240


Obtem as estatísticas de temperatura por Mês:
- Temperatura mínima
- Temperatura máxima
- Temperatura média

In [66]:
# Criar as colunas min, max, med por MES*
# Estas estatísticas já são os primeiros 3 indicadores

df_base_mes = (
    df_temperatura
        .withColumn("ano", F.year("data_medicao"))
        .withColumn("mes", F.month("data_medicao"))
)

df_stats_mes = (
    df_base_mes
    .groupBy("ano"
            ,"mes"
            ,"code_muni"
    )
    .agg(F.min("temp_media_municipio").alias("temp_min_mes")
        ,F.max("temp_media_municipio").alias("temp_max_mes")
        ,F.round(F.avg("temp_media_municipio"),2).alias("temp_media_mes")
    )
)



# print("df_stats.count()", df_stats.count())
df_stats_mes.printSchema()
df_stats_mes.filter("code_muni = 3550308.0").orderBy('ano').show(100, truncate=False)

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- temp_min_mes: double (nullable = true)
 |-- temp_max_mes: double (nullable = true)
 |-- temp_media_mes: double (nullable = true)

+----+---+---------+------------+------------+--------------+
|ano |mes|code_muni|temp_min_mes|temp_max_mes|temp_media_mes|
+----+---+---------+------------+------------+--------------+
|1995|8  |3550308.0|11.94       |19.12       |16.02         |
|1995|1  |3550308.0|18.21       |23.69       |21.44         |
|1995|7  |3550308.0|11.53       |19.33       |15.35         |
|1995|4  |3550308.0|13.42       |21.38       |17.42         |
|1995|3  |3550308.0|15.85       |21.99       |19.32         |
|1995|10 |3550308.0|11.94       |21.8        |16.25         |
|1995|6  |3550308.0|11.35       |17.61       |14.4          |
|1995|5  |3550308.0|10.7        |19.98       |15.77         |
|1995|11 |3550308.0|12.28       |21.53       |17.16         |
|19

<pre>
Todos os indicadores de clima serão inseridos em uma só tabela,
sendo assim é necessário que o valor medido (temperaturas) esteja no mesmo tipo de datatype
</pre>

In [67]:
df_stats_mes_normalize_datatype = \
    (df_stats_mes
        .withColumns({"temp_min_mes":   F.col("temp_min_mes").cast("double")
                     ,"temp_max_mes":   F.col("temp_max_mes").cast("double")
                     ,"temp_media_mes": F.col("temp_media_mes").cast("double")
                     }))

# Transformar a colunas de estatísticas em linhas
df_stats_mes_transpose = (
    df_stats_mes_normalize_datatype.select(
        "ano",
        "mes",
        "code_muni",
        F.expr("""
            stack(
                3,
                'Temperatura mínima (°C)', temp_min_mes  , 'Celsius',
                'Temperatura máxima (°C)', temp_max_mes  , 'Celsius',
                'Temperatura média (°C)' , temp_media_mes, 'Celsius'
            ) as (indicador, valor, unidade_medida)
        """)
    )
)

df_stats_mes_transpose.printSchema()

# Grava somente os últimos 10 anos
df_temp_media_mes_transpose_write = df_stats_mes_transpose.filter("ano >= 2015 and indicador = 'Temperatura média (°C)'")

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_media_ano_mes.parquet"

write_data_parquet_by_duck(df_temp_media_mes_transpose_write, write_path, prefix_file_name)


root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)



In [68]:
df_t1 = spark.read.parquet(f"{write_path}\\{prefix_file_name}")
df_t1.printSchema() # 779.800
print("Count:", df_t1.count())
df_t1.orderBy('ano','mes').limit(10).show(truncate=False)

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

Count: 779800
+----+---+---------+----------------------+-----+--------------+
|ano |mes|code_muni|indicador             |valor|unidade_medida|
+----+---+---------+----------------------+-----+--------------+
|2015|1  |2400802.0|Temperatura média (°C)|24.89|Celsius       |
|2015|1  |5215900.0|Temperatura média (°C)|23.42|Celsius       |
|2015|1  |2200053.0|Temperatura média (°C)|26.58|Celsius       |
|2015|1  |2901106.0|Temperatura média (°C)|23.97|Celsius       |
|2015|1  |4210100.0|Temperatura média (°C)|20.12|Celsius       |
|2015|1  |2201929.0|Temperatura média (°C)|24.88|Celsius       |
|2015|1  |4209854.0|Temperatura média (°C)|20.14|Celsius       |
|2015|1  |3106804.0|Temperatura média (°C)|20.1 |Celsius       |
|2015|1  |2206720.0|Temperat

Obtem as estatísticas de temperatura por ANO:
- Temperatura mínima
- Temperatura máxima
- Temperatura média
- Percentil 5%
- Percentil 90%

In [69]:
# Criar as colunas min, max, med e percentis por ANO*, que serão utilizadas para apurar os
# valores extremos (calor e frio) e os 6 indicadores de Ondas de Calor e Ondas de Frio
# * De acordo com documento elaborado por Sara Lopes de Moraes para o VERACIS

df_base = (
    df_temperatura
        .withColumn("ano", F.year("data_medicao"))
        .withColumn("mes", F.month("data_medicao"))
)

df_stats = (
    df_base
    .groupBy("ano"
            ,"code_muni"
    )
    .agg(F.min("temp_media_municipio").alias("temp_min_ano")
        ,F.max("temp_media_municipio").alias("temp_max_ano")
        ,F.avg("temp_media_municipio").alias("temp_media_ano")
        ,F.expr("percentile_approx(temp_media_municipio, 0.05)").alias("percentil_05_ano")
        ,F.expr("percentile_approx(temp_media_municipio, 0.95)").alias("percentil_95_ano")
    )
)

# print("df_stats.count()", df_stats.count())
df_stats.printSchema()
df_stats.show(10, truncate=False)

root
 |-- ano: integer (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- temp_min_ano: double (nullable = true)
 |-- temp_max_ano: double (nullable = true)
 |-- temp_media_ano: double (nullable = true)
 |-- percentil_05_ano: double (nullable = true)
 |-- percentil_95_ano: double (nullable = true)

+----+---------+------------+------------+------------------+----------------+----------------+
|ano |code_muni|temp_min_ano|temp_max_ano|temp_media_ano    |percentil_05_ano|percentil_95_ano|
+----+---------+------------+------------+------------------+----------------+----------------+
|1995|1100098.0|18.8        |26.61       |22.88742465753425 |21.12           |24.76           |
|1995|1100379.0|16.05       |27.41       |22.860082191780826|20.05           |24.86           |
|1995|1100924.0|16.44       |26.26       |22.284328767123288|19.57           |24.12           |
|1995|1101609.0|19.97       |27.0        |23.759726027397264|22.42           |25.39           |
|1995|1302108.0

In [70]:
# df_stats_ano_normalize_datatype = \
#     (df_stats
#         .withColumns({"temp_min_ano":   F.col("temp_min_ano").cast("double")
#                      ,"temp_max_ano":   F.col("temp_max_ano").cast("double")
#                      ,"temp_media_ano": F.col("temp_media_ano").cast("double")
#                      }))


# df_stats_ano_transpose = (
#     df_stats_ano_normalize_datatype.select(
#         "ano",
#         F.lit(0).alias("mes"),
#         "code_muni",
#         F.expr("""
#             stack(
#                 3,
#                 'Temperatura mínima', temp_min_ano, 'Celsius',
#                 'Temperatura máxima', temp_max_ano, 'Celsius',
#                 'Temperatura média',  temp_media_ano, 'Celsius'
#             ) as (indicador, valor, unidade_medida)
#         """)
#     )
# )

# # Grava somente os últimos 10 anos
# df_stats_ano_transpose_write = df_stats_ano_transpose.filter("ano >= 2015")

# write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
# prefix_file_name = "temperatura_min_med_max_ano.parquet"

# write_data_parquet_by_duck(df_stats_ano_transpose_write, write_path, prefix_file_name)

In [71]:
# df_t1 = spark.read.parquet(f"{write_path}\\{prefix_file_name}")
# df_t1.printSchema()
# df_t1.show(10, False)

Anexar dados estatíscos e percentis do ANO ao dado diário

Será utilizado para calcular as ondas de calor (periodos de dias consecutivos)

In [72]:
df_dia = (
    df_base.alias("base")
    .join(
        df_stats.alias("stats"),
        [
            F.col("base.ano")       == F.col("stats.ano"),
            # F.col("base.mes")       == F.col("stats.mes"),
            F.col("base.code_muni") == F.col("stats.code_muni")
        ],
        how="left",
    )
    .select(
        "stats.ano",
        "base.mes",
        "stats.code_muni",
        "stats.temp_min_ano",
        "stats.temp_max_ano",
        "stats.temp_media_ano",
        "stats.percentil_05_ano",
        "stats.percentil_95_ano",
        "base.data_medicao",
        "base.indicador",
        "base.temp_media_municipio",
        "base.unidade_medida"
    )
)

print("Número de registros no DataFrame diário:", df_dia.count()) # 151662
df_dia.printSchema()
df_dia.filter("ano = 2023 and code_muni = 3550308.0").orderBy("mes").show(10, truncate=False)
# 

Número de registros no DataFrame diário: 63676240
root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- temp_min_ano: double (nullable = true)
 |-- temp_max_ano: double (nullable = true)
 |-- temp_media_ano: double (nullable = true)
 |-- percentil_05_ano: double (nullable = true)
 |-- percentil_95_ano: double (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- indicador: string (nullable = false)
 |-- temp_media_municipio: double (nullable = true)
 |-- unidade_medida: string (nullable = false)

+----+---+---------+------------+------------+------------------+----------------+----------------+------------+----------------------+--------------------+--------------+
|ano |mes|code_muni|temp_min_ano|temp_max_ano|temp_media_ano    |percentil_05_ano|percentil_95_ano|data_medicao|indicador             |temp_media_municipio|unidade_medida|
+----+---+---------+------------+------------+------------------+---------

In [73]:
# Número de registros no DataFrame diário: 63.676.240 -> Quando juntar com a Geolocalização será > 300MM


Classifica os valores extremos para calor e frio
- Calor: temperatura diaria é maior ou igual ao percentil 95
- Frio: temperatura diária é menor ou igual ao percentil 5

In [74]:
# Inclui as colunas extremo_alto e extremo_baixo para a medição de temperatura diária
df_dia = (
    df_dia
    .withColumn(
        "extremo_alto",
        F.when(F.col("temp_media_municipio") >= F.col("percentil_95_ano")
              ,F.round((F.col("temp_media_municipio") - F.col("percentil_95_ano")),2)).otherwise(0))
    .withColumn(
        "extremo_baixo",
        F.when(F.col("temp_media_municipio") <= F.col("percentil_05_ano")
              ,F.round((F.col("percentil_05_ano") - F.col("temp_media_municipio")),2)).otherwise(0)
    )
)

In [75]:
# Criar o indicar Temperatura extrema BAIXA (°C abaixo do percentil 5)

df_dia.createOrReplaceTempView('temp_stats_diaria')

query_extremo_baixo = \
    """ Select ano
              ,mes
              ,code_muni
              ,'Temperatura extrema baixa (°C abaixo do percentil 5)' as indicador
              ,min(extremo_baixo) valor
              ,'Celsius' as unidade_medida
          from temp_stats_diaria
         group by all
    """

df_extremo_baixo_mensal = \
    spark.sql(query_extremo_baixo)

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_extrema_baixa_ano_mes.parquet"

write_data_parquet_by_duck(df_extremo_baixo_mensal, write_path, prefix_file_name)

In [76]:
# Criar o indicar Temperatura extrema ALTA (°C acima do percentil 95)
query_extremo_alto = \
    """ Select ano
              ,mes
              ,code_muni
              ,'Temperatura extrema alta (°C acima do percentil 95)' as indicador
              ,max(extremo_alto) valor
              ,'Celsius' as unidade_medida
          from temp_stats_diaria
         group by all
    """

df_extremo_alto_mensal = \
    spark.sql(query_extremo_alto)

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_extrema_alta_ano_mes.parquet"

write_data_parquet_by_duck(df_extremo_alto_mensal, write_path, prefix_file_name)


Separa somente os dias quentes ou frios de acordo com a regra definida no passo anterior

In [77]:

df_flag = \
    (df_dia.withColumn("flag_dia_quente"
                     ,F.when(F.col("temp_media_municipio") > F.col("percentil_95_ano"), 1).otherwise(0))
           .withColumn("flag_dia_frio"
                      ,F.when(F.col("temp_media_municipio") < F.col("percentil_05_ano"), 1).otherwise(0)))

# Mantém apenas os dias que atenderam ao critério de dias com Extremos quentes ou frios
df_quentes = df_flag.filter(F.col("flag_dia_quente") == 1)
df_frios   = df_flag.filter(F.col("flag_dia_frio")   == 1)

# Janela ordenada por data para cada ponto geográfico, assim será possível identificar os períodos consecutivos de ondas de calor ou frio
janela_loc = Window.partitionBy("code_muni").orderBy("data_medicao")

# Ao subtrair a ordem do registro (rn) da data, dias consecutivos geram
# exatamente o mesmo identificador de grupo (grupo_id)
df_eventos = \
    (df_quentes
        .withColumn("rn", F.row_number().over(janela_loc)) \
        .withColumn("grupo_id", F.expr("date_sub(data_medicao, CAST(rn AS INT))")))

df_eventos_frio = \
    (df_frios
        .withColumn("rn", F.row_number().over(janela_loc))
        .withColumn("grupo_id", F.expr("date_sub(data_medicao, CAST(rn AS INT))"))
)


In [78]:
# df_eventos.count() # 3.111.745
df_eventos.filter("code_muni = 3550308.0").orderBy('ano', 'mes').show(100,False)

+----+---+---------+------------+------------+------------------+----------------+----------------+------------+----------------------+--------------------+--------------+------------+-------------+---------------+-------------+---+----------+
|ano |mes|code_muni|temp_min_ano|temp_max_ano|temp_media_ano    |percentil_05_ano|percentil_95_ano|data_medicao|indicador             |temp_media_municipio|unidade_medida|extremo_alto|extremo_baixo|flag_dia_quente|flag_dia_frio|rn |grupo_id  |
+----+---+---------+------------+------------+------------------+----------------+----------------+------------+----------------------+--------------------+--------------+------------+-------------+---------------+-------------+---+----------+
|1995|1  |3550308.0|10.7        |23.69       |17.308849315068493|12.31           |21.99           |1995-01-06  |Temperatura diaria °C |22.63               |Celsius       |0.64        |0.0          |1              |0            |1  |1995-01-05|
|1995|1  |3550308.0|10.7

In [79]:
# Identificação dos Eventos de CALOR e Validação da Duração (Global, sem quebra de mês/ano, usando apenas o município e o grup_id, 
# que separa a ondas (agrupamento de dias consecuticos))

df_eventos_duracao = (
    df_eventos
    .groupBy("code_muni", "grupo_id")
    .agg(
        F.count("data_medicao").alias("duracao_total_onda"),
        F.min("data_medicao").alias("inicio_onda"),
        F.max("data_medicao").alias("fim_onda")
    )
    .filter(F.col("duracao_total_onda") > 2) # Filtra apenas eventos reais (> 2 dias de duração)
)

# 2. Retornar os DIAS INDIVIDUAIS das ondas de calor válidas mantendo os metadados da onda
df_dias_em_onda = (
    df_eventos
    .join(df_eventos_duracao, ["code_muni", "grupo_id"], "inner")
    .select("code_muni", 
            "data_medicao", 
            "ano", 
            "mes", 
            "grupo_id", 
            "duracao_total_onda",
            "temp_media_municipio"
    )
)



In [80]:
df_dias_em_onda.filter('code_muni = 3550308.0').show(100,False)

+---------+------------+----+---+----------+------------------+--------------------+
|code_muni|data_medicao|ano |mes|grupo_id  |duracao_total_onda|temp_media_municipio|
+---------+------------+----+---+----------+------------------+--------------------+
|3550308.0|2008-01-01  |2008|1  |2007-05-14|5                 |22.87               |
|3550308.0|2007-12-31  |2007|12 |2007-05-14|5                 |24.2                |
|3550308.0|2007-12-30  |2007|12 |2007-05-14|5                 |23.36               |
|3550308.0|2007-12-29  |2007|12 |2007-05-14|5                 |24.64               |
|3550308.0|2007-12-28  |2007|12 |2007-05-14|5                 |23.88               |
|3550308.0|1997-11-14  |1997|11 |1997-10-01|3                 |22.3                |
|3550308.0|1997-11-13  |1997|11 |1997-10-01|3                 |23.09               |
|3550308.0|1997-11-12  |1997|11 |1997-10-01|3                 |23.12               |
|3550308.0|2004-11-06  |2004|11 |2004-05-18|3                 |22

In [81]:
# Identificação dos Eventos de FRIO e Validação da Duração (Global, sem quebra de mês/ano, usando apenas as coordenadas geográfica)
df_eventos_duracao_frio = (
    df_eventos_frio
    .groupBy("code_muni", "grupo_id")
    .agg(
        F.count("data_medicao").alias("duracao_total_onda"),
        F.min("data_medicao").alias("inicio_onda"),
        F.max("data_medicao").alias("fim_onda")
    )
    .filter(F.col("duracao_total_onda") > 2) # Filtra apenas eventos reais (> 2 dias)
)

# 2. Dias individuais de ondas de frio válidas
df_dias_em_onda_frio = (
    df_eventos_frio
    .join(df_eventos_duracao_frio, ["code_muni", "grupo_id"], "inner")
    .select(
        "code_muni", 
        "data_medicao", 
        "ano", 
        "mes", 
        "grupo_id", 
        "duracao_total_onda",
        "temp_media_municipio"  # Mão dupla: temperatura para amplitude e magnitude
    )
)

#### Adicionar as métricas:

<pre>
- Número de ondas de calor (N-OdC)      : Total de eventos de ondas de calor registrados em um determinado ano.
- Frequência das ondas de calor (F-OdC) : Número total de dias que compõem as ondas de calor ao longo do ano.
- Duração das ondas de calor (D-OdC)    : Duração, em dias, do evento de onda de calor mais longo registrado no ano.
- Amplitude das ondas de calor (A-OdC)  : Maior valor da temperatura média diária observado durante eventos de onda de calor no ano.
- Magnitude das ondas de calor (M-OdC)  : Média da temperatura média diária considerando todos os dias de ocorrência de ondas de calor no ano.
</pre>

In [82]:
# Métricas MENSAIS para Ondas de CALOR

df_metricas_mensal = (
    df_dias_em_onda
    .groupBy("code_muni", "ano", "mes")
    .agg(
        # N-OdC: Número de ondas distintas no mês
        F.countDistinct("grupo_id").alias("numero_ondas_calor"),
        
        # F-OdC: Total exato de dias sob onda de calor DENTRO do mês
        F.count("data_medicao").alias("frequencia_dias_onda_calor"),
        
        # D-OdC: Duração máxima (e média) das ondas no mês
        F.max("duracao_total_onda").alias("duracao_maxima_onda"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas"),
        
        # A-OdC: Maior temperatura diária observada em dias de onda de calor no mês
        F.max("temp_media_municipio").alias("amplitude_onda_calor"),
        
        # M-OdC: Média das temperaturas diárias considerando os dias de onda de calor no mês
        F.round(F.avg("temp_media_municipio"), 2).alias("magnitude_onda_calor")
    )
    .orderBy("ano", "mes", "code_muni")
)


df_metricas_mensal.printSchema()



root
 |-- code_muni: double (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- numero_ondas_calor: long (nullable = false)
 |-- frequencia_dias_onda_calor: long (nullable = false)
 |-- duracao_maxima_onda: long (nullable = true)
 |-- duracao_media_ondas: double (nullable = true)
 |-- amplitude_onda_calor: double (nullable = true)
 |-- magnitude_onda_calor: double (nullable = true)



In [83]:
# Normaliza o dataype da métricas, elas serão persistidas em uma mesma tabela

df_metricas_mensal_normalize_datatype = \
    (df_metricas_mensal
        .withColumns({"numero_ondas_calor":         F.round(F.col("numero_ondas_calor").cast("double"),2)
                     ,"frequencia_dias_onda_calor": F.round(F.col("frequencia_dias_onda_calor").cast("double"),2)
                     ,"duracao_maxima_onda":        F.round(F.col("duracao_maxima_onda").cast("double"),2)
                     ,"amplitude_onda_calor":       F.round(F.col("amplitude_onda_calor").cast("double"),2)
                     ,"amplitude_onda_calor":       F.round(F.col("amplitude_onda_calor").cast("double"),2)
                     }))


df_metricas_mensal_transpose = (
    df_metricas_mensal_normalize_datatype.select(
        "ano",
        "mes",
        "code_muni",
        F.expr("""
            stack(
                5,
                'Número de ondas de calor'      , numero_ondas_calor        , 'eventos',
                'Frequência das ondas de calor' , frequencia_dias_onda_calor, 'dias',
                'Duração das ondas de calor'    , duracao_maxima_onda       , 'dias',
                'Amplitude das ondas de calor'  , amplitude_onda_calor      , 'Celsius',
                'Magnitude das ondas de calor'  , magnitude_onda_calor      , 'Celsius'
            ) as (indicador                     , valor                     , unidade_medida)
        """)
    )
)

df_metricas_mensal_transpose.printSchema()

# df_metricas_mensal_transpose.show(truncate=False)

# Grava somente os últimos 10 anos
df_metricas_mensal_transpose_write = df_metricas_mensal_transpose.filter("ano >= 2015")

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_ondas_calor_ano_mes.parquet"


write_data_parquet_by_duck(df_metricas_mensal_transpose_write, write_path, prefix_file_name)


root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)



In [84]:
df_OC_ano_mes = spark.read.parquet(f"{write_path}\\{prefix_file_name}")
df_OC_ano_mes.printSchema()
df_OC_ano_mes.filter("code_muni = 3550308").orderBy("ano","mes").show(100,False)

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+----+---+---------+-----------------------------+-----+--------------+
|ano |mes|code_muni|indicador                    |valor|unidade_medida|
+----+---+---------+-----------------------------+-----+--------------+
|2015|1  |3550308.0|Número de ondas de calor     |1.0  |eventos       |
|2015|1  |3550308.0|Frequência das ondas de calor|3.0  |dias          |
|2015|1  |3550308.0|Duração das ondas de calor   |3.0  |dias          |
|2015|1  |3550308.0|Amplitude das ondas de calor |25.16|Celsius       |
|2015|1  |3550308.0|Magnitude das ondas de calor |24.34|Celsius       |
|2016|1  |3550308.0|Número de ondas de calor     |1.0  |eventos       |
|2016|1  |3550308.0|Frequência das ondas de calor|2.0  |dias          |
|2016|1  |3550308.0|Duração das ondas

In [85]:
# Métricas MENSAIS para Ondas de FRIO

df_metricas_mensal_frio = (
    df_dias_em_onda_frio
    .groupBy("code_muni", "ano", "mes")
    .agg(
        # N-OdF: Número de ondas de frio distintas
        F.countDistinct("grupo_id").alias("numero_ondas_frio"),
        
        # F-OdF: Frequência total de dias em onda de frio no mês
        F.count("data_medicao").alias("frequencia_dias_onda_frio"),
        
        # D-OdF: Duração máxima (e média) das ondas no mês
        F.max("duracao_total_onda").alias("duracao_maxima_onda_frio"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas_frio"),
        
        # A-OdF: Menor temperatura diária observada durante as ondas no mês
        F.min("temp_media_municipio").alias("amplitude_onda_frio"),
        
        # M-OdF: Média das temperaturas diárias durante os dias de onda de frio
        F.round(F.avg("temp_media_municipio"), 2).alias("magnitude_onda_frio")
    )
    .orderBy("ano", "mes", "code_muni")
)

df_metricas_mensal_frio.printSchema()

df_metricas_mensal_frio.filter("code_muni = 3550308.0").limit(100).show(truncate=False)

# write_data(df_metricas_mensal_frio, ROOT_DATA_PATH, "Ondas_Frio_Mensal")

# df_metricas_mensal_frio.filter("latitude = -23 and longitude = -46").show(10, truncate=False)


root
 |-- code_muni: double (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- numero_ondas_frio: long (nullable = false)
 |-- frequencia_dias_onda_frio: long (nullable = false)
 |-- duracao_maxima_onda_frio: long (nullable = true)
 |-- duracao_media_ondas_frio: double (nullable = true)
 |-- amplitude_onda_frio: double (nullable = true)
 |-- magnitude_onda_frio: double (nullable = true)

+---------+----+---+-----------------+-------------------------+------------------------+------------------------+-------------------+-------------------+
|code_muni|ano |mes|numero_ondas_frio|frequencia_dias_onda_frio|duracao_maxima_onda_frio|duracao_media_ondas_frio|amplitude_onda_frio|magnitude_onda_frio|
+---------+----+---+-----------------+-------------------------+------------------------+------------------------+-------------------+-------------------+
|3550308.0|1995|5  |1                |3                        |3                       |3.0        

In [86]:
df_metricas_mensal_frio_normalize_datatype = \
    (df_metricas_mensal_frio
        .withColumns({"numero_ondas_frio":          F.round(F.col("numero_ondas_frio").cast("double"),2)
                     ,"frequencia_dias_onda_frio":  F.round(F.col("frequencia_dias_onda_frio").cast("double"),2)
                     ,"duracao_maxima_onda_frio":   F.round(F.col("duracao_maxima_onda_frio").cast("double"),2)
                     ,"amplitude_onda_frio":        F.round(F.col("amplitude_onda_frio").cast("double"),2)
                     ,"amplitude_onda_frio":        F.round(F.col("amplitude_onda_frio").cast("double"),2)
                     }))

df_metricas_mensal_frio_transpose = (
    df_metricas_mensal_frio_normalize_datatype.select(
        "ano",
        "mes",
        "code_muni",
        F.expr("""
            stack(5
                 ,'Número de ondas de frio'     , numero_ondas_frio         , 'eventos'
                 ,'Frequência das ondas de frio', frequencia_dias_onda_frio , 'dias'
                 ,'Duração das ondas de frio'   , duracao_maxima_onda_frio  , 'dias'
                 ,'Amplitude das ondas de frio' , amplitude_onda_frio       , 'Celsius'
                 ,'Magnitude das ondas de frio' , magnitude_onda_frio       , 'Celsius'
            ) as (indicador                     , valor                     , unidade_medida)
        """)
    )
)

df_metricas_mensal_frio_transpose.printSchema()

# Grava somente os últimos 10 anos
df_metricas_mensal_frio_transpose_write = df_metricas_mensal_frio_transpose.filter("ano >= 2015")

write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
prefix_file_name = "temperatura_ondas_frio_ano_mes.parquet"

write_data_parquet_by_duck(df_metricas_mensal_frio_transpose_write, write_path, prefix_file_name)


root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)



In [87]:
df_OF_ano_mes = spark.read.parquet(f"{write_path}\\{prefix_file_name}")
df_OF_ano_mes.printSchema()
# df_OF_ano_mes.filter("code_muni = 3550308 and unidade_medida = 'eventos' and cast(valor as int) > 1 ").orderBy("ano","mes").show(100,False)
df_OF_ano_mes.filter("code_muni = 3550308 and ano = 2021 ").orderBy("ano","mes").show(100,False)

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+----+---+---------+----------------------------+-----+--------------+
|ano |mes|code_muni|indicador                   |valor|unidade_medida|
+----+---+---------+----------------------------+-----+--------------+
|2021|6  |3550308.0|Número de ondas de frio     |1.0  |eventos       |
|2021|6  |3550308.0|Frequência das ondas de frio|1.0  |dias          |
|2021|6  |3550308.0|Duração das ondas de frio   |4.0  |dias          |
|2021|6  |3550308.0|Amplitude das ondas de frio |8.44 |Celsius       |
|2021|6  |3550308.0|Magnitude das ondas de frio |8.44 |Celsius       |
|2021|7  |3550308.0|Número de ondas de frio     |3.0  |eventos       |
|2021|7  |3550308.0|Frequência das ondas de frio|11.0 |dias          |
|2021|7  |3550308.0|Duração das ondas de frio  

In [88]:
# # Métricas ANUAIS para Ondas de CALOR

# df_metricas_anual = (
#     df_dias_em_onda
#     .groupBy("code_muni", "ano")
#     .agg(
#         # N-OdC: Número de ondas distintas no ano
#         F.countDistinct("grupo_id").alias("numero_ondas_calor"),
        
#         # F-OdC: Total de dias do ano passados sob onda de calor
#         F.count("data_medicao").alias("frequencia_dias_onda_calor"),
        
#         # D-OdC: Duração máxima do evento no ano
#         F.max("duracao_total_onda").alias("duracao_maxima_onda"),
#         F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas"),
        
#         # A-OdC: Maior temperatura média diária observada durante ondas no ano
#         F.max("temp_media_municipio").alias("amplitude_onda_calor"),
        
#         # M-OdC: Média das temperaturas diárias em todos os dias de onda de calor no ano
#         F.round(F.avg("temp_media_municipio"), 2).alias("magnitude_onda_calor")
#     )
#     .orderBy("ano", "code_muni")
# )

# df_metricas_anual.printSchema()

# df_metricas_anual.filter("code_muni = 3550308.0").show(10,False)


In [89]:
# df_metricas_anual_normalize_datatype = \
#     (df_metricas_anual
#         .withColumns({"numero_ondas_calor":         F.round(F.col("numero_ondas_calor").cast("double"),2)
#                      ,"frequencia_dias_onda_calor": F.round(F.col("frequencia_dias_onda_calor").cast("double"),2)
#                      ,"duracao_maxima_onda":        F.round(F.col("duracao_maxima_onda").cast("double"),2)
#                      ,"amplitude_onda_calor":       F.round(F.col("amplitude_onda_calor").cast("double"),2)
#                      ,"amplitude_onda_calor":       F.round(F.col("amplitude_onda_calor").cast("double"),2)
#                      }))

# df_metricas_anual_transpose = (
#     df_metricas_anual_normalize_datatype.select(
#         "ano",
#         F.lit(0).alias("mes"),
#         "code_muni",
#         F.expr("""
#             stack(
#                 5,
#                 'Número de ondas de calor'      , numero_ondas_calor        , 'eventos'
#                ,'Frequência das ondas de calor' , frequencia_dias_onda_calor, 'dias'
#                ,'Duração das ondas de calor'    , duracao_maxima_onda       , 'dias'
#                ,'Amplitude das ondas de calor'  , amplitude_onda_calor      , 'Celsius'
#                ,'Magnitude das ondas de calor'  , magnitude_onda_calor      , 'Celsius'
#             ) as (indicador                     , valor                     , unidade_medida)

#         """)
#     )
# )

# df_metricas_anual_transpose.printSchema()

# # Grava somente os últimos 10 anos
# df_metricas_anual_transpose_write = df_metricas_anual_transpose.filter("ano >= 2015")

# write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
# prefix_file_name = "temperatura_ondas_calor_ano.parquet"

# write_data_parquet_by_duck(df_metricas_anual_transpose_write, write_path, prefix_file_name)

In [90]:
# df_OC_ano = spark.read.parquet(f"{write_path}\\{prefix_file_name}")
# df_OC_ano.printSchema()
# df_OC_ano.filter("code_muni = 3550308 and ano = 2021 ").orderBy("ano","mes").show(100,False)

In [91]:
# # Métricas anuais para Ondas de FRIO

# df_metricas_anual_frio = (
#     df_dias_em_onda_frio
#     .groupBy("code_muni", "ano")
#     .agg(
#         # N-OdF: Número de ondas de frio distintas no ano
#         F.countDistinct("grupo_id").alias("numero_ondas_frio"),
        
#         # F-OdF: Total de dias do ano passados sob onda de frio
#         F.count("data_medicao").alias("frequencia_dias_onda_frio"),
        
#         # D-OdF: Duração máxima (e média) do evento no ano
#         F.max("duracao_total_onda").alias("duracao_maxima_onda_frio"),
#         F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas_frio"),
        
#         # A-OdF: Menor temperatura diária registrada durante ondas de frio no ano
#         F.min("temp_media_municipio").alias("amplitude_onda_frio"),
        
#         # M-OdF: Média das temperaturas nos dias sob onda de frio no ano
#         F.round(F.avg("temp_media_municipio"), 2).alias("magnitude_onda_frio")
#     )
#     .orderBy("ano", "code_muni")
# )

# df_metricas_anual_frio.printSchema()

In [92]:
# df_metricas_anual_frio_normalize_datatype = \
#     (df_metricas_anual_frio
#         .withColumns({"numero_ondas_frio":          F.round(F.col("numero_ondas_frio").cast("double"),2)
#                      ,"frequencia_dias_onda_frio":  F.round(F.col("frequencia_dias_onda_frio").cast("double"),2)
#                      ,"duracao_maxima_onda_frio":   F.round(F.col("duracao_maxima_onda_frio").cast("double"),2)
#                      ,"amplitude_onda_frio":        F.round(F.col("amplitude_onda_frio").cast("double"),2)
#                      ,"amplitude_onda_frio":        F.round(F.col("amplitude_onda_frio").cast("double"),2)
#                      }))

# df_metricas_ano_frio_transpose = (
#     df_metricas_anual_frio_normalize_datatype.select(
#         "ano",
#         F.lit(0).alias("mes"),
#         "code_muni",
#         F.expr("""
#             stack(
#                 5,
#                 'Número de ondas de frio', numero_ondas_frio, 'eventos',
#                 'Frequência das ondas de frio', frequencia_dias_onda_frio, 'dias',
#                 'Duração das ondas de frio', duracao_maxima_onda_frio, 'dias',
#                 'Amplitude das ondas de frio', amplitude_onda_frio, 'Celsius',
#                 'Magnitude das ondas de frio', magnitude_onda_frio, 'Celsius'
#             ) as (indicador, valor, unidade_medida)
#         """)
#     )
# )

# df_metricas_ano_frio_transpose.printSchema()

# # Grava somente os últimos 10 anos
# df_metricas_ano_frio_transpose_write = df_metricas_ano_frio_transpose.filter("ano >= 2015")

# write_path       = r"{ROOT_DATA_PATH}consolidados\indicador_temperatura".format(ROOT_DATA_PATH = ROOT_DATA_PATH)
# prefix_file_name = "temperatura_ondas_frio_ano.parquet"

# write_data_parquet_by_duck(df_metricas_ano_frio_transpose_write, write_path, prefix_file_name)

In [93]:
# df_OF_ano = spark.read.parquet(f"{write_path}\\{prefix_file_name}")
# df_OF_ano.printSchema()
# df_OF_ano.filter("code_muni = 3550308 and ano = 2021 ").orderBy("ano","mes").show(100,False)

In [97]:
# "C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_media_ano_mes.parquet"
# "C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_extrema_baixa_ano_mes.parquet"
# "C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_extrema_alta_ano_mes.parquet"
# "C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_ondas_calor_ano_mes.parquet"
# "C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_ondas_frio_ano_mes.parquet"



df_temperatura_media_ano_mes 		 = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_media_ano_mes.parquet")
df_temperatura_media_ano_mes.createOrReplaceTempView('tb_temperatura_media_ano_mes')

df_temperatura_extrema_baixa_ano_mes = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_extrema_baixa_ano_mes.parquet")
df_temperatura_extrema_baixa_ano_mes.createOrReplaceTempView('tb_temperatura_extrema_baixa_ano_mes')

df_temperatura_extrema_alta_ano_mes  = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_extrema_alta_ano_mes.parquet")
df_temperatura_extrema_alta_ano_mes.createOrReplaceTempView('tb_temperatura_extrema_alta_ano_mes')

df_temperatura_ondas_calor_ano_mes   = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_ondas_calor_ano_mes.parquet")
df_temperatura_ondas_calor_ano_mes.createOrReplaceTempView('tb_temperatura_ondas_calor_ano_mes')

df_temperatura_ondas_frio_ano_mes 	 = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\consolidados\indicador_temperatura\temperatura_ondas_frio_ano_mes.parquet")
df_temperatura_ondas_frio_ano_mes.createOrReplaceTempView('tb_temperatura_ondas_frio_ano_mes')



In [101]:

df_temperatura_ondas_frio_ano_mes.printSchema()

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)



In [99]:
query = \
    """with ind_clima as (
            Select * from tb_temperatura_media_ano_mes
            Union All
            Select * from tb_temperatura_extrema_baixa_ano_mes
            Union All 
            Select * from tb_temperatura_extrema_alta_ano_mes
            Union All
            Select * from tb_temperatura_ondas_calor_ano_mes
            Union All
            Select * from tb_temperatura_ondas_frio_ano_mes
            )
        Select * 
          from ind_clima
         where 1=1
           --and ano = 2025
           --and mes = 6
           --and code_muni = 3550308.0
           --and indicador rlike 'Temperatura'
         order by mes, indicador
    """
df_result = spark.sql(query)

In [ ]:
df_result.count() # 5.985.625

5985625